# Atividade 3 - Trabalhando com dados NaN

1 - Crie um arquivo no colab.

2 - Abra o dataset de vinhos utilizado em aulas passadas.

3 - Faça os passos de predição de NaN numérico, codificação de valores categóricos e separação do dataset entre treino e teste.

4 - Imprima em PDF o arquivo do colab e entregue a atividade no Classroom.

In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

In [3]:
path = 'https://raw.githubusercontent.com/OrlandoIFPR/ml2/main/Dataframes/'
wine_reviews = pd.read_csv(path+ 'winemag-data-130k-v2.csv')
display(wine_reviews.head())

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


---
## Etapa 1 - Diagnostico dos valores ausentes

In [ ]:
nan = pd.DataFrame({
    "nulos": wine_reviews.isnull().sum(),
    "percentual": (wine_reviews.isnull().mean() * 100).round(2)
})
display(nan.sort_values("nulos", ascending=False))

print("Linhas:", wine_reviews.shape[0], "| Colunas:", wine_reviews.shape[1])

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(wine_reviews.isnull(), cbar=False, cmap="viridis")
plt.title("Mapa dos valores ausentes")
plt.show()

**Leitura:** `region_2` (~61%) e `designation` (~29%) tem NaN demais para serem uteis. A unica coluna **numerica** com NaN e `price` (~6,9%) - e ela que sera **predita**. `country`, `province` e `variety` tem pouquissimos NaN.

---
## Etapa 2 - Limpeza inicial

Descarto colunas sem valor preditivo (indice, textos livres, identificadores) e as de NaN excessivo; removo as poucas linhas sem `country`/`variety`.

In [ ]:
df = wine_reviews.drop(columns=[
    "Unnamed: 0",              # apenas o indice do CSV
    "description", "title",    # texto livre
    "taster_twitter_handle",   # redundante com taster_name
    "designation", "region_2"  # NaN em excesso
])

df = df.dropna(subset=["country", "province", "variety"])

print("Formato apos limpeza:", df.shape)
display(df.isnull().sum())

---
## Etapa 3 - Tratamento dos NaN categoricos

`region_1` e `taster_name` nao podem ser preditos de forma confiavel e nao sao o alvo da atividade; NaN aqui e informacao ("nao informado"), entao viram uma categoria propria.

In [ ]:
df["region_1"] = df["region_1"].fillna("Desconhecido")
df["taster_name"] = df["taster_name"].fillna("Desconhecido")

print("NaN restantes:")
display(df.isnull().sum())

---
## Etapa 4 - Codificacao dos valores categoricos

Os modelos do scikit-learn so aceitam numeros. Verifico primeiro a **cardinalidade** de cada coluna categorica, pois ela define a tecnica.

In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
display(df[cat_cols].nunique().sort_values(ascending=False).to_frame("categorias"))

**Decisao:** `winery` (~16 mil), `region_1` (~1,2 mil) e `variety` (~700) tem cardinalidade altissima. One-Hot criaria milhares de colunas esparsas, entao uso **OrdinalEncoder** (1 coluna por atributo), adequado a modelos baseados em arvore, que serao usados na predicao do preco.

Abaixo, um exemplo de **One-Hot** aplicado a `taster_name` (baixa cardinalidade), so para comparacao das duas tecnicas.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

df_enc = df.copy()

encoder = OrdinalEncoder()
df_enc[cat_cols] = encoder.fit_transform(df_enc[cat_cols])

display(df_enc.head())
df_enc.dtypes

In [ ]:
# Comparacao: One-Hot em uma coluna de baixa cardinalidade
exemplo_onehot = pd.get_dummies(df["taster_name"], prefix="taster")
print("Colunas geradas pelo One-Hot:", exemplo_onehot.shape[1])
display(exemplo_onehot.head())

---
## Etapa 5 - Predicao do NaN numerico (`price`)

Em vez de preencher com a media (que achata a distribuicao), treino um **RandomForestRegressor** nas linhas que tem preco e uso o modelo para prever as que nao tem.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

com_preco  = df_enc[df_enc["price"].notna()]
sem_preco  = df_enc[df_enc["price"].isna()]

print("Com preco:", com_preco.shape[0], "| Sem preco:", sem_preco.shape[0])

X_p = com_preco.drop(columns=["price"])
y_p = com_preco["price"]

In [ ]:
# Validacao do imputador antes de usa-lo
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

imputador = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
imputador.fit(Xp_tr, yp_tr)

pred = imputador.predict(Xp_te)
print("MAE :", round(mean_absolute_error(yp_te, pred), 2))
print("R2  :", round(r2_score(yp_te, pred), 3))
print("MAE da media (baseline):",
      round(mean_absolute_error(yp_te, [yp_tr.mean()] * len(yp_te)), 2))

**Leitura:** o erro do modelo e bem menor que o da media, ou seja, `points`, `country`, `variety` e `winery` realmente carregam informacao sobre o preco. O imputador esta validado.

In [ ]:
# Treino final com todos os dados conhecidos e preenchimento dos NaN
imputador.fit(X_p, y_p)
precos_preditos = imputador.predict(sem_preco.drop(columns=["price"]))

df_enc.loc[df_enc["price"].isna(), "price"] = precos_preditos

print("NaN restantes em price:", df_enc["price"].isna().sum())
print("Total de NaN no dataset:", df_enc.isna().sum().sum())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["price"].dropna(), bins=60, ax=ax[0])
ax[0].set(title="Antes (somente precos originais)", xlim=(0, 300), xlabel="Preco")

sns.histplot(df_enc["price"], bins=60, ax=ax[1], color="darkorange")
ax[1].set(title="Depois (com valores preditos)", xlim=(0, 300), xlabel="Preco")

plt.tight_layout()
plt.show()

**Leitura:** a forma da distribuicao (assimetrica a direita) foi preservada. Um `fillna(media)` teria criado um pico artificial em ~35, distorcendo a variavel.

---
## Etapa 6 - Separacao entre treino e teste

Alvo: `points` (nota do vinho). Atributos: todo o restante ja codificado e sem NaN.

In [ ]:
X = df_enc.drop(columns=["points"])
y = df_enc["points"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Treino:", X_train.shape, y_train.shape)
print("Teste :", X_test.shape,  y_test.shape)
print("Proporcao de teste:", round(len(X_test) / len(X) * 100, 1), "%")

In [ ]:
display(X_train.head())
print("Media de points - treino:", round(y_train.mean(), 2),
      "| teste:", round(y_test.mean(), 2))

---
## Conclusao

1. **Diagnostico:** `region_2` e `designation` descartadas por excesso de NaN; `price` identificada como a unica numerica com ausencias.
2. **NaN categorico:** preenchido com a categoria `"Desconhecido"`, preservando a informacao da ausencia.
3. **Codificacao:** `OrdinalEncoder` pela alta cardinalidade das colunas (One-Hot geraria milhares de colunas esparsas).
4. **Predicao do NaN numerico:** `RandomForestRegressor` validado contra o baseline da media, com erro menor e distribuicao original preservada.
5. **Split:** 80% treino / 20% teste com `random_state=42` para reprodutibilidade.

Dataset final sem nenhum NaN e inteiramente numerico, pronto para o treinamento de modelos.